In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/ ev_charging_data /delta_table")
print(f"Total stations: {df.count()}")

Total stations: 237751


In [0]:
from pyspark.sql.functions import count, desc

top_countries = df.groupBy("country_code") \
    .agg(count("id").alias("station_count")) \
    .orderBy(desc("station_count")) \
    .limit(10)

top_countries.show()

+------------+-------------+
|country_code|station_count|
+------------+-------------+
|          US|        81361|
|          GB|        26627|
|          DE|        23243|
|          ES|        17795|
|          CA|        16347|
|          FR|        12945|
|          IT|         9934|
|          NL|         7977|
|          SE|         4892|
|          NO|         4662|
+------------+-------------+



In [0]:
power_dist = df.groupBy("power_class") \
    .agg(count("id").alias("count")) \
    .orderBy(desc("count"))

power_dist.show()

+------------------+------+
|       power_class| count|
+------------------+------+
|    AC_L1_(<7.5kW)|107143|
| AC_HIGH_(22-49kW)| 55541|
|DC_FAST_(50-149kW)| 37327|
|  AC_L2_(7.5-21kW)| 24237|
|DC_ULTRA_(>=150kW)| 13503|
+------------------+------+



In [0]:
from pyspark.sql.functions import round, avg, col

fast_dc_summary = df.groupBy("is_fast_dc") \
    .agg(
        count("id").alias("station_count"),
        round(avg("power_kw"), 2).alias("avg_power_kw"),
        round(avg("ports"), 2).alias("avg_ports")
    )

fast_dc_summary.show()

+----------+-------------+------------+---------+
|is_fast_dc|station_count|avg_power_kw|avg_ports|
+----------+-------------+------------+---------+
|      true|        50830|       124.3|     2.92|
|     false|       186921|       11.04|     1.69|
+----------+-------------+------------+---------+



In [0]:
top_cities = df.filter(col("city") != "Unknown City") \
    .groupBy("city", "country_code") \
    .agg(count("id").alias("station_count")) \
    .orderBy(desc("station_count")) \
    .limit(10)

from pyspark.sql.functions import col
top_cities.show()

+-----------+------------+-------------+
|       city|country_code|station_count|
+-----------+------------+-------------+
|     London|          GB|         7441|
|Los Angeles|          US|         2122|
|Hammersmith|          GB|         1117|
|   Montréal|          CA|          991|
|     Berlin|          DE|          803|
|    Toronto|          CA|          761|
|  San Diego|          US|          753|
|    Atlanta|          US|          678|
|     Austin|          US|          647|
|    Hamburg|          DE|          604|
+-----------+------------+-------------+



In [0]:
avg_power_country = df.groupBy("country_code") \
    .agg(
        round(avg("power_kw"), 2).alias("avg_power_kw"),
        count("id").alias("station_count")
    ) \
    .filter(col("station_count") > 100) \
    .orderBy(desc("avg_power_kw")) \
    .limit(15)

avg_power_country.show()

+------------+------------+-------------+
|country_code|avg_power_kw|station_count|
+------------+------------+-------------+
|          IN|      898.24|         1170|
|          KR|      230.71|          161|
|          TW|      205.25|          162|
|          IL|      112.65|          291|
|          UA|      110.59|          555|
|          TR|       85.48|         1191|
|          RU|       76.77|         2194|
|          LT|       72.83|          764|
|          AU|       69.05|         1219|
|          HR|       63.83|          265|
|          RO|       63.42|          402|
|          JP|        60.7|         1606|
|          EE|       57.25|          169|
|          SI|       56.77|          171|
|          RS|       56.74|          106|
+------------+------------+-------------+

